# NYT + Open Library End-to-End Smoke Test

This notebook tests the full pipeline for one NYT published date:
1. Pull NYT bestsellers for a single date.
2. Upsert into `nyt_entries`.
3. Enrich each ISBN13 via Open Library.
4. Validate joined output from `nyt_entries` + `openlibrary_enrichment`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

project_root = Path('/Users/zacurbiztondo/literary-analysis')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import settings
from src.ingest.http import HttpClient
from src.ingest.nyt import NytClient, NytConfig
from src.ingest.openlibrary import OpenLibraryClient, OpenLibraryConfig
from src.ingest.repo import Repo
from src.utils.io import connect_sqlite

settings.ensure_dirs()

http = HttpClient(
    cache_path=settings.http_cache_path,
    expire_seconds=settings.http_cache_expire_seconds,
    contact_email=settings.contact_email,
)

nyt = NytClient(http=http, cfg=NytConfig(api_key=settings.nyt_api_key, rps=settings.nyt_rps))
openlibrary = OpenLibraryClient(http=http, cfg=OpenLibraryConfig(rps=settings.openlibrary_rps))

conn = connect_sqlite(settings.db_path)
repo = Repo(conn)
repo.init_schema()

print(f"DB path: {settings.db_path}")


In [ ]:
# Pick one NYT published date for the smoke test.
# Change this value as needed.
published_date = '2024-06-16'

entries = nyt.fetch_lists_for_date(published_date)
upserts = repo.upsert_nyt_entries(entries)

print(f"Fetched NYT entries: {len(entries)}")
print(f"Upserted NYT entries: {upserts}")

pd.read_sql_query(
    """
    SELECT list_name, published_date, rank, title, author, isbn13
    FROM nyt_entries
    WHERE published_date = ?
    ORDER BY list_name, rank
    LIMIT 30
    """,
    conn,
    params=(published_date,),
)


In [ ]:
# Enrich only ISBN13s from the chosen NYT date.
isbn_df = pd.read_sql_query(
    """
    SELECT DISTINCT isbn13
    FROM nyt_entries
    WHERE published_date = ?
      AND isbn13 IS NOT NULL
      AND TRIM(isbn13) <> ''
    ORDER BY isbn13
    """,
    conn,
    params=(published_date,),
)

isbn_values = isbn_df['isbn13'].tolist()
print(f"Distinct ISBN13s to enrich: {len(isbn_values)}")

rows = []
failures = 0
for isbn13 in isbn_values:
    row = openlibrary.fetch_isbn13_work(isbn13)
    rows.append(row)
    if row.last_error:
        failures += 1

repo.upsert_openlibrary_enrichment(rows)
print(f"Enrichment rows upserted: {len(rows)}")
print(f"Enrichment failures: {failures}")


In [ ]:
# Validate final join for the selected date.
enriched = pd.read_sql_query(
    """
    SELECT
        n.list_name,
        n.published_date,
        n.rank,
        n.title,
        n.author,
        n.isbn13,
        e.work_key,
        e.subjects,
        e.subject_places,
        e.description,
        e.last_error,
        e.last_checked_at
    FROM nyt_entries n
    LEFT JOIN openlibrary_enrichment e
      ON e.isbn13 = n.isbn13
    WHERE n.published_date = ?
    ORDER BY n.list_name, n.rank
    """,
    conn,
    params=(published_date,),
)

enriched.head(50)


In [ ]:
# Optional quality checks
summary = pd.read_sql_query(
    """
    SELECT
      COUNT(*) AS rows_for_date,
      SUM(CASE WHEN e.isbn13 IS NOT NULL THEN 1 ELSE 0 END) AS matched_enrichment_rows,
      SUM(CASE WHEN e.description IS NOT NULL AND TRIM(e.description) <> '' THEN 1 ELSE 0 END) AS with_description,
      SUM(CASE WHEN e.subjects IS NOT NULL AND TRIM(e.subjects) <> '' THEN 1 ELSE 0 END) AS with_subjects,
      SUM(CASE WHEN e.subject_places IS NOT NULL AND TRIM(e.subject_places) <> '' THEN 1 ELSE 0 END) AS with_subject_places,
      SUM(CASE WHEN e.last_error IS NOT NULL AND TRIM(e.last_error) <> '' THEN 1 ELSE 0 END) AS with_errors
    FROM nyt_entries n
    LEFT JOIN openlibrary_enrichment e
      ON e.isbn13 = n.isbn13
    WHERE n.published_date = ?
    """,
    conn,
    params=(published_date,),
)
summary


In [ ]:
conn.close()
